# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** The `@id` of each entity (record set, field, or column) is used for referencing in all subsequent steps.

In [ ]:
# List all record sets and their fields, showing @id for each.
print("Record sets found in the dataset:\n")
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Below, we demonstrate extraction for all record sets found in Section 2. You can select specific record sets and field `@id`s as needed.

In [ ]:
# Extract all record set @ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("RecordSet IDs to load:", record_sets_ids)

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}")
    print()
# For demonstration, pick the first record set for further analysis
if record_sets_ids:
    primary_record_set = record_sets_ids[0]
    print(f"Using record set: {primary_record_set}")
    print(dataframes[primary_record_set].head())
else:
    primary_record_set = None
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Replace the field `@id` variables with IDs observed in the overview above. The code example below assumes a numeric field (e.g., coverage, peptide_count, molecular weight) is present.


In [ ]:
import numpy as np

if primary_record_set is not None:
    df = dataframes[primary_record_set].copy()

    # Choose a numeric field by its @id (update this with a valid @id from Section 2)
    # Example: suppose the field @id for "molecular weight" is 'cr:mw' and for "description" is 'cr:description'
    numeric_field_id = None
    group_field_id = None

    # Try to auto-select a numeric field
    for col in df.columns:
        # Heuristic: column contains 'weight' or 'coverage' or 'count' or is float/int
        if df[col].dtype.kind in 'iuf':
            numeric_field_id = col
            break
    else:
        # Try by name pattern
        for col in df.columns:
            if 'weight' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower():
                numeric_field_id = col
                break

    if numeric_field_id is None:
        print('No numeric field found in primary_record_set!')
    else:
        print(f"Using numeric field: {numeric_field_id}")

        # Try to select a likely grouping field, fallback to None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break

        # Remove missing values
        df_dropna = df.dropna(subset=[numeric_field_id])
        threshold = df_dropna[numeric_field_id].quantile(0.75)  # 75th percentile as threshold
        filtered_df = df_dropna[df_dropna[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped data by {group_field_id} (showing mean {numeric_field_id}):")
            print(grouped_df.head())
else:
    print("No record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Update `numeric_field_id` and `group_field_id` as appropriate.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        # Plot mean value per group
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.title(f"Top 20 mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Utilized `mlcroissant` to access Mass Spectrometry Analysis data of extracellular vesicles from stimulated human mast cells.
- Loaded available record sets and fields using their `@id` for robust, schema-driven access.
- Performed extraction, exploratory filtering, and basic visualizations over numeric fields (e.g., protein coverage/weight/count) where available.

You can continue deeper analysis, such as outlier investigation, correlation analysis, or predictive modeling using the extracted pandas DataFrame(s).